In [3]:
import pandas as pd

rfm = pd.read_csv('../data/rfm_base.csv')
print(rfm.shape)
rfm.head()

(5878, 4)


,customer_id,recency,frequency,monetary
0,12346,325 days 02:49:00,12,77556.46
1,12347,1 day 20:58:00,8,4921.53
2,12348,74 days 23:37:00,5,2019.40
3,12349,18 days 02:59:00,4,4428.69
4,12350,309 days 20:49:00,1,334.40


In [4]:
print(rfm['recency'].dtype)
print(rfm['recency'].head())

object
0    325 days 02:49:00
1       1 day 20:58:00
2     74 days 23:37:00
3     18 days 02:59:00
4    309 days 20:49:00
Name: recency, dtype: object


In [5]:
# Convert recency string to a clean integer (days only)
rfm['recency'] = pd.to_timedelta(rfm['recency']).dt.days
print(rfm['recency'].dtype)
rfm.head()

int64


,customer_id,recency,frequency,monetary
0,12346,325,12,77556.46
1,12347,1,8,4921.53
2,12348,74,5,2019.40
3,12349,18,4,4428.69
4,12350,309,1,334.40


In [6]:
# Score R, F, M into quintiles (1-5)
# Recency: lower is better, so we reverse the labels
rfm['R_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])

# Frequency and Monetary: higher is better
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])

rfm[['customer_id', 'recency', 'R_score', 'frequency', 'F_score', 'monetary', 'M_score']].head(10)

,customer_id,recency,R_score,frequency,F_score,monetary,M_score
0,12346,325,2,12,5,77556.46,5
1,12347,1,5,8,4,4921.53,5
2,12348,74,3,5,4,2019.40,4
3,12349,18,5,4,3,4428.69,5
4,12350,309,2,1,1,334.40,2
5,12351,374,2,1,1,300.93,2
6,12352,35,4,10,5,2849.84,4
7,12353,203,2,2,2,406.76,2
8,12354,231,2,1,1,1079.40,3
9,12355,213,2,2,2,947.61,3


In [8]:
# Combine R, F, M scores into a single RFM score string
rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

# Calculate an average score for simpler segmentation
rfm['RFM_avg'] = (rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)) / 3

# Assign segment names based on R and F scores (the classic RFM segmentation logic)
def segment_customer(row):
    r = int(row['R_score'])
    f = int(row['F_score'])
    
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r <= 2 and f >= 4:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['segment'] = rfm.apply(segment_customer, axis=1)

# See segment distribution
rfm['segment'].value_counts()

segment
Lost               1523
Champions          1482
Loyal Customers    1221
Needs Attention     856
New Customers       443
At Risk             353
Name: count, dtype: int64

In [9]:
# Revenue contribution by segment
segment_summary = rfm.groupby('segment').agg(
    customer_count=('customer_id', 'count'),
    total_revenue=('monetary', 'sum'),
    avg_revenue=('monetary', 'mean')
).sort_values('total_revenue', ascending=False)

segment_summary['pct_of_total_revenue'] = (segment_summary['total_revenue'] / rfm['monetary'].sum() * 100).round(1)
segment_summary

,customer_count,total_revenue,avg_revenue,pct_of_total_revenue
segment,,,,
Champions,1482,12024330.14,8113.583090,69.2
Loyal Customers,1221,2510046.34,2055.730008,14.4
At Risk,353,1090694.27,3089.785467,6.3
Needs Attention,856,703039.78,821.308154,4.0
Lost,1523,654426.70,429.695798,3.8
New Customers,443,392267.02,885.478600,2.3


In [10]:
# Export the full RFM table with segments for Power BI
rfm.to_csv('../data/rfm_segmented.csv', index=False)
print("Saved rfm_segmented.csv")

Saved rfm_segmented.csv
